### Core metrics
---------------
Compute and print core descriptive statistics for the advanced A1 dataset.

In [2]:
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH = "../data/raw/dataset_mood_smartphone.csv"  
ID_COL    = "id"
TIME_COL  = "time"
VAR_COL   = "variable"
VAL_COL   = "value"
# ─────────────────────────────────────────────────────────────────────────────


def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=[TIME_COL])
    return df


def compute_metrics(df: pd.DataFrame) -> dict:
    # Per-patient record counts
    records_per_patient = df.groupby(ID_COL).size()

    # Per-patient observation span in days
    def obs_days(g):
        span = g[TIME_COL].max() - g[TIME_COL].min()
        return span.days + 1  # +1 to include both start and end days

    days_per_patient = df.groupby(ID_COL).apply(obs_days, include_groups=False)

    metrics = {
        "n_raw_records":              len(df),
        "n_unique_patients":          df[ID_COL].nunique(),
        "n_unique_variables":         df[VAR_COL].nunique(),
        "n_unique_timestamps":        df[TIME_COL].nunique(),
        "date_range_start":           df[TIME_COL].min(),
        "date_range_end":             df[TIME_COL].max(),
        "avg_records_per_patient":    records_per_patient.mean(),
        "median_records_per_patient": records_per_patient.median(),
        "avg_obs_days_per_patient":   days_per_patient.mean(),
    }
    return metrics


def print_metrics(m: dict) -> None:
    print("=" * 50)
    print("DATASET CORE METRICS")
    print("=" * 50)
    print(f"  Raw records              : {m['n_raw_records']:,}")
    print(f"  Unique patients          : {m['n_unique_patients']:,}")
    print(f"  Unique variables         : {m['n_unique_variables']:,}")
    print(f"  Unique timestamps        : {m['n_unique_timestamps']:,}")
    print(f"  Date range               : {m['date_range_start'].date()} → {m['date_range_end'].date()}")
    print(f"  Avg records / patient    : {m['avg_records_per_patient']:.1f}")
    print(f"  Median records / patient : {m['median_records_per_patient']:.1f}")
    print(f"  Avg observation days     : {m['avg_obs_days_per_patient']:.1f}")
    print("=" * 50)


if __name__ == "__main__":
    df = load_data(DATA_PATH)
    metrics = compute_metrics(df)
    print_metrics(metrics)

DATASET CORE METRICS
  Raw records              : 376,912
  Unique patients          : 27
  Unique variables         : 19
  Unique timestamps        : 336,907
  Date range               : 2014-02-17 → 2014-06-09
  Avg records / patient    : 13959.7
  Median records / patient : 14581.0
  Avg observation days     : 79.4


#### Why this matters


#### Notes

**Observation days** is computed as `max(timestamp) ` −  `min(timestamp)` per patient in calendar days.

**Unique timestamps** counts distinct datetime values across the whole dataset. Useful for flagging how densely sampled the data is before we aggregate to daily averages for the target variable


#### Additional Analysis

Median is often **more informative** than mean in real data.

Consider your dataset. Suppose a patient's daily screen time over 5 days is:

```
10, 15, 12, 11, 180  (minutes)
```
```python
mean   = 45.6  # suggests heavy usage every day
median = 12    # reflects the typical day accurately
```

The mean tells you "on average 45 min" which is misleading. The median tells you "on a typical day, 12 min" which is true.

**Median is especially critical in data analysis for:**

- **Outlier detection** — if mean and median diverge a lot, you have skew or outliers worth investigating
- **Reporting typical behavior** — e.g. "the typical patient used social apps for X min/day"
- **Imputation decisions** — median imputation is more robust than mean imputation for skewed variables
- **Sanity checks** — comparing mean vs median is a quick way to assess distribution shape


> *"For skewed variables such as screen time and app usage, we use the median as a more robust central tendency measure, as the mean is sensitive to the extreme values observed in this dataset."*